In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
import base64
load_dotenv()

/backup/workspace/github/agentic-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
with open("blood_work.png", "rb") as f:
    image_b64=base64.b64encode(f.read()).decode()
    

iVBORw0KGgoAAAANSUhEUgAAAmwAAAHgCAIAAACXbaZMAACxv0lEQVR4nOzdeVwT1944/iGBkASIIKuyBBAQIahYqiBuiCjWrVi8eK+KYlVEqdvFBQtq64obVvsgUhatD9XSihuLFatYZRNUtA0iYF0AZZEiEBICSeb3ejy/O9+52QiRzfp5/5WcOXPmzDnDfJiZkzka


In [9]:
llm=ChatGroq(
    model="qwen/qwen3.6-27b",              
)

message = HumanMessage(content=[
    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
    {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
])

response = llm.invoke([message])
print(response.content)


<think>
The user wants me to extract data from a blood work report image.

**1. Identify the sections:**
- Patient Info: Rajesh Sharma, Age 48, Male, Date: May 7, 2026
- COMPLETE BLOOD COUNT (CBC)
- LIPID PANEL
- METABOLIC PANEL
- LIVER FUNCTION (This section is cut off, but the header is visible).

**2. Extract data point by point:**

**COMPLETE BLOOD COUNT (CBC)**
- Hemoglobin: 15.1 g/dL. Normal range: 13.5 - 17.5. -> **Normal**
- Hematocrit: 44%. Normal range: 41 - 53%. -> **Normal**
- WBC: 6.8 x 10^3/uL. Normal range: 4.5 - 11.0. -> **Normal**
- Platelets: 220 x 10^3/uL. Normal range: 150 - 400. -> **Normal**

**LIPID PANEL**
- Total Cholesterol: 238 mg/dL. Normal range: < 200. -> **High (Abnormal)**
- LDL Cholesterol: 162 mg/dL. Normal range: < 100. -> **High (Abnormal)**
- HDL Cholesterol: 36 mg/dL. Normal range: > 40. -> **Low (Abnormal)**
- Triglycerides: 188 mg/dL. Normal range: < 150. -> **High (Abnormal)**

**METABOLIC PANEL**
- Glucose (Fasting): 92 mg/dL. Normal range: 70

In [13]:
from langchain.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver


@tool
def get_diet_recommendtaions(condition: str)->dict:
    """Given a health condition, returns a diet plan. Condition must of one of: normal, high_cholesterol, high_sugar."""

    diet_plans = {
        "hight_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

SYSTEM_PROMPT="""
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan."""

diat_agent=create_agent(
    llm,
    tools=[get_diet_recommendtaions],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)

In [14]:
config={"configurable":{"thread_id":"user-alice-session-3"}}

result=diat_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."}
        ])]    
},
    config=config)
print(result["messages"][-1].content)

Based on the blood work report provided, here is the analysis of the test results:

**COMPLETE BLOOD COUNT (CBC)**
*   **Hemoglobin:** 15.1 g/dL (Normal: 13.5 - 17.5) - **Normal**
*   **Hematocrit:** 44% (Normal: 41 - 53%) - **Normal**
*   **WBC:** 6.8 x10^3/uL (Normal: 4.5 - 11.0) - **Normal**
*   **Platelets:** 220 x10^3/uL (Normal: 150 - 400) - **Normal**

**LIPID PANEL**
*   **Total Cholesterol:** 238 mg/dL (Normal: <200) - **High**
*   **LDL Cholesterol:** 162 mg/dL (Normal: <100) - **High**
*   **HDL Cholesterol:** 36 mg/dL (Normal: >40) - **Low**
*   **Triglycerides:** 188 mg/dL (Normal: <150) - **High**

**METABOLIC PANEL**
*   **Glucose (Fasting):** 92 mg/dL (Normal: 70 - 99) - **Normal**
*   **HbA1c:** 5.3% (Normal: <5.7%) - **Normal**
*   **Creatinine:** 1.0 mg/dL (Normal: 0.7 - 1.3) - **Normal**
*   **eGFR:** 82 mL/min (Normal: >60) - **Normal**

**Summary of Findings:**
The patient has normal blood cell counts and kidney function. However, the lipid panel indicates signifi